## Statistics of Lyman-α profiles, metal emission lines and absorption lines

This notebook generates histograms of key parameters in the megatable

In [ ]:
from astropy.io import fits
import astropy.table as aptb
import matplotlib.pyplot as plt

from tangelo.statistics import generate_histogram, make_scatter

# Load the megatab
SPEC_SOURCE = "R21"  #  Where are the spectra from? 'R21' for Richard+21, 'APER' for apertures
SPEC_TYPE   = "weight_skysub"

tabfile = f"megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_absv_ubs_{SPEC_TYPE}.fits"
tabhdul = fits.open(tabfile)
megatable = aptb.Table(tabhdul[1].data)

# Only keep sources with significant Lyman alpha detection
megatab = megatable[(megatable['SNRR'] > 5.0) | (megatable['SNRB'] > 5.0)]

In [ ]:
# First plot some basic histograms of source parameters: z, Lya lum, magnification

dp_mask = megatab['SNRB'] > 3.0

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
fig.subplots_adjust(wspace=0)

generate_histogram('z', megatab, ax=axes[0], bins=20, masks = [dp_mask, ~dp_mask], labels = [r'DP', r'SP'],
                   colors = ['mediumslateblue', 'indianred'], alpha=0.4, 
                   hist_kwargs=[{'edgecolor': 'black', 'hatch': '//'}, {'edgecolor': 'black'}])
# axes[0].set_xlabel("Redshift (z)")
axes[0].set_title("Redshift Distribution")

generate_histogram('LYA_LUM', megatab, ax=axes[1], bins=20, masks = [dp_mask, ~dp_mask],
                   colors = ['mediumslateblue', 'indianred'], alpha=0.4, 
                   hist_kwargs=[{'edgecolor': 'black', 'hatch': '//'}, {'edgecolor': 'black'}])
# axes[1].set_xlabel("Lyman Alpha Luminosity (erg/s)")
axes[1].set_title(r"Ly$\alpha$ Luminosity Distribution")
axes[1].set_ylabel("")

generate_histogram('MU', megatab, ax=axes[2], bins=20, logify=True, masks = [dp_mask, ~dp_mask],
                   colors = ['mediumslateblue', 'indianred'], alpha=0.4, 
                   hist_kwargs=[{'edgecolor': 'black', 'hatch': '//'}, {'edgecolor': 'black'}])
# axes[2].set_xlabel("Magnification")
axes[2].set_title("Magnification Distribution")
axes[2].set_ylabel("")

# plt.tight_layout()
fig.savefig("./plots/basic_source_properties.pdf", dpi=300)
plt.show()


In [ ]:
# Plot the relative offsets of absorption and emission lines in velocity space relative to Lyman alpha
import numpy as np

truemitters = megatab['VFLUX_SYS'] / megatab['VFLUX_SYS_ERR'] > 3.0
li_absorbers  = megatab['EW_LI_ABS']  / megatab['EW_LI_ABS_ERR']  < -3.0
hi_absorbers  = megatab['EW_HI_ABS']  / megatab['EW_HI_ABS_ERR']  < -3.0
tot_absorbers = megatab['EW_TOT_ABS'] / megatab['EW_TOT_ABS_ERR'] < -3.0
truabsorbers = li_absorbers | hi_absorbers | tot_absorbers

print(f"Sources with any significant absorption:              {np.sum(truabsorbers)} / {len(megatab)}")
print(f"  Low-ionisation stack  (LI,  SiII+CII):             {np.sum(li_absorbers)}")
print(f"  High-ionisation stack (HI,  SiIV):                 {np.sum(hi_absorbers)}")
print(f"  Both LI and HI:                                    {np.sum(li_absorbers & hi_absorbers)}")
print(f"  Total stack           (TOT, LI+HI):                {np.sum(tot_absorbers)}")
print(f"  TOT only (not LI or HI):                           {np.sum(tot_absorbers & ~li_absorbers & ~hi_absorbers)}")

# Count individual line detections (SNR < -3.0)
abs_lines = ['SiII1260', 'CII1334', 'SiIV1394', 'SiIV1403']
total_line_detections = 0
print(f"\nIndividual absorption line detections (SNR < -3.0):")
for line in abs_lines:
    col = f'SNR_{line}'
    n = int(np.sum(megatab[col] < -3.0))
    total_line_detections += n
    print(f"  {line:<12}: {n}")
print(f"  {'Total':<12}: {total_line_detections}")

def get_abs_vel_offset(tab):
    """Calculate velocity offset of absorption lines from Lyman alpha by selecting the strongest absorption line and returning
    its velocity offset"""
    dv_stacks = np.array([tab['DV_LI_ABS'], tab['DV_HI_ABS'], tab['DV_TOT_ABS']], dtype=float)
    err_stacks = np.array([tab['DV_LI_ABS_ERR'], tab['DV_HI_ABS_ERR'], tab['DV_TOT_ABS_ERR']], dtype=float)

    # Mask errors where DV is NaN (no valid measurement for that stack)
    err_stacks[np.isnan(dv_stacks)] = np.nan

    # Identify sources with no valid measurement in any stack
    all_nan = np.all(np.isnan(err_stacks), axis=0)

    # Temporarily fill all-NaN columns so nanargmin doesn't raise
    err_stacks_filled = err_stacks.copy()
    err_stacks_filled[:, all_nan] = 0.0

    # Select the stack with the smallest error (most precise measurement)
    min_idx = np.nanargmin(np.abs(err_stacks_filled), axis=0)

    # Gather the selected DV values
    result = dv_stacks[min_idx, np.arange(len(tab))]

    # Restore NaN for sources with no detection in any stack
    result[all_nan] = np.nan

    return result


bins = np.linspace(-700,600,14)
fig, axs = plt.subplots(1,2,figsize=(8,4), facecolor='w', sharey=True)
fig.subplots_adjust(wspace=0)
deltav_abs = get_abs_vel_offset(megatab)
ax = axs[1]
ax.hist(deltav_abs[truabsorbers], bins = bins, label='Absorption lines',
        **{'alpha': 0.4, 'color': 'olivedrab', 'edgecolor': 'black', 'hatch': '//'})
deltav_em = -megatab['DELTAV_LYA']
ax.hist(deltav_em[truemitters], bins = bins, label='Emission lines',
        **{'alpha': 0.4, 'color':'firebrick', 'edgecolor': 'black'})
ax.set_xlabel(r"$\Delta V$ (from red)")

# Calculate the velocity offset from the blue peak for double-peaked sources using the separation in wavelength between the blue and red peaks
br_sep = megatab['LPEAKR'] - megatab['LPEAKB']
# Convert to rest-frame wavelength
br_sep_rest = br_sep / (1 + megatab['z'])
# Convert to velocity space
br_deltav = br_sep_rest / 1215.67 * 3e5
# The velocity offset from the blue peak is then the velocity offset from the red peak minus the separation between the peaks
deltav_em_blue = deltav_em - br_deltav
deltav_abs_blue = deltav_abs - br_deltav

ax = axs[0]
bins = np.linspace(0,1100,12)
ax.hist(-deltav_em_blue[truemitters], bins = bins, label='Emission lines',
        **{'alpha': 0.4, 'color':'firebrick', 'edgecolor': 'black'})
ax.hist(-deltav_abs_blue[truabsorbers], bins = bins, label='Absorption lines',
        **{'alpha': 0.4, 'color': 'olivedrab', 'edgecolor': 'black', 'hatch': '//'})
ax.set_xlabel(r"$\Delta V$ (from blue)")
ax.set_ylabel(f"N")
ax.legend(loc=2)
fig.savefig(f"./plots/lya_velocity_offsets.pdf", dpi=300)
plt.show()
plt.close()

In [ ]:
#now make a scatter plot showing both mu and abs mag for absorbers and non-absorbers:
from matplotlib.gridspec import GridSpec
from astropy.cosmology import Planck18
import astropy.units as u

def scatter_with_hist(x, y, xerr=None, yerr=None, xbins=None, ybins=None, 
                     fig=None, axes=None, color='blue', label=None, alpha=0.6,
                     xlabel='X values', ylabel='Y values', title=None, density=False):
    """
    Create or overlay a scatter plot with histograms along the x and y axes.
    
    Parameters:
    -----------
    x, y : array-like
        Data points to be plotted
    xerr, yerr : array-like, optional
        Error bars for x and y values
    xbins, ybins : int or array-like, optional
        Bins for the x and y histograms
    fig : matplotlib Figure, optional
        Existing figure to plot on
    axes : tuple of matplotlib Axes, optional
        Existing axes (scatter, histx, histy) to plot on
    color : str, optional
        Color for this dataset
    label : str, optional
        Label for the legend
    alpha : float, optional
        Transparency level
    xlabel, ylabel : str, optional
        Axis labels
    title : str, optional
        Plot title
    """
    
    # Create new figure if none provided
    if fig is None or axes is None:
        fig = plt.figure(figsize=(4, 4),facecolor='w')
        gs = GridSpec(4, 4)
        
                # Main scatter plot
        ax_scatter = fig.add_subplot(gs[1:4, 0:3])
        
        # X histogram (top)
        ax_histx = fig.add_subplot(gs[0, 0:3], sharex=ax_scatter)
        
        # Y histogram (right)
        ax_histy = fig.add_subplot(gs[1:4, 3], sharey=ax_scatter)
        
        # Adjust spacing between subplots
        plt.subplots_adjust(wspace=0.05, hspace=0.05)
        
        # Remove tick labels from histograms completely
        ax_histx.tick_params(axis="x", labelbottom=False, bottom=True)
        ax_histy.tick_params(axis="y", labelleft=False, left=True)
        
        # Configure tick marks (keep ticks but remove labels)
        ax_histx.tick_params(axis="both", which='both', 
                           direction='in', length=3,
                           labelbottom=False, labelleft=False)
        ax_histy.tick_params(axis="both", which='both',
                           direction='in', length=3,
                           labelleft=False, labelbottom=False)
        
        # Set labels only on scatter plot
        ax_scatter.set_xlabel(xlabel)
        ax_scatter.set_ylabel(ylabel)
        if title:
            ax_histx.set_title(title)
        
        axes = (ax_scatter, ax_histx, ax_histy)
    else:
        ax_scatter, ax_histx, ax_histy = axes
    
    # Plot the scatter points
    scatter = ax_scatter.scatter(x, y, alpha=alpha, color=color, label=label)
    
    # Add error bars if provided
    if xerr is not None or yerr is not None:
        xerr = xerr if xerr is not None else np.zeros_like(x)
        yerr = yerr if yerr is not None else np.zeros_like(y)
        ax_scatter.errorbar(x, y, xerr=xerr, yerr=yerr, 
                          fmt='none', ecolor=color, alpha=alpha*0.7)
    
    # Plot the histograms
    ax_histx.hist(x, bins=xbins if xbins is not None else 20, 
                 color=color, alpha=alpha, density=density)
    ax_histy.hist(y, bins=ybins if ybins is not None else 20, 
                 orientation='horizontal', color=color, alpha=alpha, density=density)
    
    # Ensure histograms are flush with scatter plot
#     ax_histx.set_xlim(ax_scatter.get_xlim())
#     ax_histy.set_ylim(ax_scatter.get_ylim())
    
    # Add legend if labels are provided
    if label is not None:
        ax_scatter.legend()
    
    return fig, axes

def app2abs(ap, z, mu):
    """Calculates the absolute magnitude from the apparent magnitude, redshift, and magnification."""
    _map = ap + 2.5 * np.log10(mu)
    _d = Planck18.luminosity_distance(z)
    _mab = _map - 5 * np.log10((_d / (10 * u.pc)).decompose()).value
    return _mab

# Use this function to plot the relationship between magnification and absolute magnitude for the sources in the megatable, 
# highlighting those with significant absorption lines

absmags = app2abs(megatab['MAG_AUTO_HST_F814W'], megatab['z'], megatab['MU'])
absmagsmax = app2abs(megatab['MAG_AUTO_HST_F814W'] + megatab['MAGERR_AUTO_HST_F814W'], megatab['z'], megatab['MU'] + megatab['MU_ERR'])
absmagsmin = app2abs(megatab['MAG_AUTO_HST_F814W'] - megatab['MAGERR_AUTO_HST_F814W'], megatab['z'], megatab['MU'] - megatab['MU_ERR'])
absmagerrl = absmags - absmagsmin
absmagerru = absmagsmax - absmags


mask_sig = (np.abs(np.nansum([megatab['DV_LI_ABS'], megatab['DV_HI_ABS'], megatab['DV_TOT_ABS']], axis=0)) > 0) \
            & (megatab['z'] < 6.7) & (megatab['z'] > 2.9) \
            & (np.logical_and(-30 < absmags, absmags < -10))
mask_supsig = (np.abs(np.nansum([megatab['DV_LI_ABS'], megatab['DV_HI_ABS'], megatab['DV_TOT_ABS']], axis=0)) > 0) \
            & (megatab['z'] < 6.7) & (megatab['z'] > 2.9) \
            & (np.logical_and(-30 < absmags, absmags < -10)) \
            & (megatab['DELTAV_LYA'] > -np.inf)
mask_insig = (np.nansum([megatab['DV_LI_ABS'], megatab['DV_HI_ABS'], megatab['DV_TOT_ABS']], axis=0) == 0) \
            & (megatab['z'] < 6.7) & (megatab['z'] > 2.9) \
            & (np.logical_and(-30 < absmags, absmags < -10))


fig, axes = scatter_with_hist(megatab['MU'][mask_sig], absmags[mask_sig], 
                              xerr=megatab['MU_ERR'][mask_sig], yerr=[absmagerrl[mask_sig], absmagerru[mask_sig]], 
                              fig=fig, color='maroon', alpha=0.8, xbins=np.geomspace(1,100,11), xlabel=r'$\mu$', 
                              ylabel='F814W luminosity [AB mag.]', ybins = np.linspace(-23,-13,10))
axes[0].scatter(megatab['MU'][mask_supsig], absmags[mask_supsig],
                edgecolors='orange', alpha=0.8, marker='o', s=16, facecolors='none')
scatter_with_hist(megatab['MU'][mask_insig], absmags[mask_insig], 
                  xerr=megatab['MU_ERR'][mask_insig], yerr=megatab['MAGERR_AUTO_HST_F814W'][mask_insig], 
                  fig=fig, axes=axes, color='slategrey', alpha=0.5, xbins=np.geomspace(1,100,10),
                 ybins = np.linspace(-23,-13,11))
# best_mask = mask_sig * (megatab['DELTAV_LYA'] > -np.inf)
# scatter_with_hist(megatab['MU'][best_mask], absmags[best_mask], 
#                   xerr=megatab['MU_ERR'][best_mask], yerr=megatab['MAGERR_AUTO_HST_F814W'][best_mask], 
#                   fig=fig, axes=axes, color='darkorange', alpha=0.5, xbins=np.geomspace(1,100,10),
#                  ybins = np.linspace(-23,-13,11))
axes[0].set_xscale('log')
axes[0].invert_yaxis()
axes[0].axhline(np.nanmedian(absmags[mask_insig]), color='slategrey', linestyle='--', alpha=0.6, label='median')
axes[0].axvline(np.nanmedian(megatab['MU'][mask_insig]), color='slategrey', linestyle='--', alpha=0.6)
axes[0].legend()
fig.savefig(f"./plots/abs_mags_mus.pdf", bbox_inches='tight')
plt.show()

In [ ]:
#SCATTER PLOTS OF LINE RATIOS FROM VERIFIED EMITTER SPECTRA
import glob
import error_propagation as ep

def make_eparr(arr1, arr2):
    """Create an array of Complex numbers from two arrays of values and errors."""
    return np.array([ep.Complex(x,y) for (x,y) in zip(arr1, arr2)])

def make_ratios(tab, lines1, lines2, ew=False):
    """Make the ratio of the sum of the fluxes in lines1 to the sum of the fluxes in lines2, propagating errors 
    and optionally dividing by the continuum to get equivalent widths."""
    f1 = np.sum([make_eparr(tab[f"FLUX_{line}"], tab[f"FLUX_ERR_{line}"]) for line in lines1], axis=0)
#     goodrows1 = np.array([x.value >= 3 * x.error for x in f1])
    f2 = np.sum([make_eparr(tab[f"FLUX_{line}"], tab[f"FLUX_ERR_{line}"]) for line in lines2], axis=0)
#     goodrows2 = np.array([x.value >= 3 * x.error for x in f2])
    if ew:
        f1 /= make_eparr(tab[f"CONT_{lines1[0]}"], tab[f"CONT_ERR_{lines1[0]}"])
        f2 /= make_eparr(tab[f"CONT_{lines2[0]}"], tab[f"CONT_ERR_{lines2[0]}"])
#     goodrows = goodrows1 * goodrows2
    rat = f1 / f2
    f1up = np.sum([tab[f"FLUX_UB_{line}"] for line in lines1], axis=0)
    ratup = f1up / f2
    if ew:
        ratup *= np.nan
    # Make a mask for where the ratio is significant (both numerator and denominator are > 3 sigma)
    signifnums = np.array([x.value > 3 * x.error for x in f1])
    signifdoms = np.array([x.value > 3 * x.error for x in f2])
    return [rat, ratup, signifnums, signifdoms]

def get_gutkin_flux(file_path, pattern):
    """
    Sum all columns whose headers contain a given pattern.
    
    Args:
        file_path (str): Path to the data file with ## headers
        pattern (str): Pattern to match in column headers (e.g., '[OIII]')
    
    Returns:
        np.ndarray: Sum of matching columns (shape: [n_rows])
    """
    # Load data (skip ## header)
    data = np.loadtxt(file_path, skiprows=1)
    
    # Load and clean headers
    with open(file_path) as f:
        headers = [h.strip() for h in f.readline().lstrip('#').split()]
    
    # Find matching columns
    matching_cols = [i for i, h in enumerate(headers) if h[:len(pattern)] == pattern]
    
    if not matching_cols:
        raise ValueError(f"No columns matched pattern '{pattern}'. Available headers: {headers}")
    
    # Sum across matching columns (axis=1 for row-wise sum)
    return np.sum(data[:, matching_cols], axis=1)

ratplots = [
    [[['CIII1907','CIII1909'], ['OIII1666']],
     [['SiIII1883', 'SiIII1892'], ['CIII1907','CIII1909']]],
    [[['HeII1640'], ['OIII1666']],
     [['CIV1548', 'CIV1551'], ['CIII1907','CIII1909']]],
    [[['NV1238', 'NV1243'], ['HeII1640']],
     [['CIII1907','CIII1909'], ['HeII1640']]],
    [[['CIV1548', 'CIV1551'], ['CIII1907','CIII1909']],
     [['CIV1548', 'CIV1551'], ['HeII1640']]],
]
xlabs = ['SiIII/CIII', 'CIV/CIII', 'CIII/HeII', 'CIV/HeII']
ylabs = ['CIII/OIII', 'HeII/OIII', 'NV/HeII', 'CIV/CIII']
nlr_files = glob.glob(f"../BEAGLE/BEAGLE-general/templates/AGN/nlr_*r?.fits")
linealiases = {
    'CIV1548': '@1549',
    'CIII1907': '@1909',
    'SiIII1883': '@1888',
    'NV1238': '@1240',
    'HeII1640': 'HeBaA@1640',
    'OIII1660': '@1665'
}

diagrams = [
    ['SiIII1883', 'CIII1907', 'CIII1907', 'OIII1660'],
    ['CIV1548', 'CIII1907', 'HeII1640', 'OIII1660'], 
    ['CIII1907', 'HeII1640', 'NV1238', 'HeII1640'],   
    ['CIV1548', 'HeII1640', 'CIV1548', 'CIII1907'],    
]

lims = [
    {'ylims': [0., 4.5], 'xlims': [0.0, 2.0]},
    {'ylims': [0., 4.5], 'xlims': [0.0, 5.0]},
    {'ylims': [0., 3.0], 'xlims': [0.0, 7.0]},
    {'ylims': [0., 3.0], 'xlims': [0.0, 7.0]}
]

truemitters = megatab['VFLUX_SYS'] / megatab['VFLUX_SYS_ERR'] > 3.0

fig, axs = plt.subplots(2,2,figsize=(6,6), facecolor='w')
fig.subplots_adjust(wspace=0.25, hspace=0.2)
axf = axs.flatten()
for i, (rats, ax) in enumerate(zip(ratplots, axf)):
    _l = diagrams[i]
    raty = make_ratios(megatab[truemitters],*rats[0])
    ratx = make_ratios(megatab[truemitters],*rats[1])
#     signifs = np.product(np.array([np.array([x.value > 3 * x.error for x in y]) for y in [raty[0],ratx[0]]]), axis=0).astype(bool)
    alllines = [rats[0][0],rats[0][1],rats[1][0],rats[1][1]]
    significances = np.array([
        (np.sum([megatab[truemitters][f"FLUX_{line}"] for line in r], axis=0) 
                > 3 * np.sqrt(np.sum(np.square([megatab[truemitters][f"FLUX_ERR_{line}"] for line in r]), axis=0))) \
        * np.prod([megatab[truemitters][f"FLAG_{line}"] == '' for line in r], axis=0)
            for r in alllines
    ])
    tsig = np.array(["".join(str(int(v)) for v in significances[:,i]) for i in range(len(significances[0]))])
    #detections
    detections = ratx[2] & ratx[3] & raty[2] & raty[3]
    make_scatter([np.array([x.value for x in ratx[0]]), np.array([x.error for x in ratx[0]])], 
                   [np.array([x.value for x in raty[0]]), np.array([x.error for x in raty[0]])],
                   ax, vmin=2.9, vmax=6.7,
                   mask = (megatab[truemitters]['SNRB'] > 3.0) & detections, edgecolor='slateblue', label='',
                  alpha=0.8, alpha_e=0.8)
    make_scatter([np.array([x.value for x in ratx[0]]), np.array([x.error for x in ratx[0]])], 
                   [np.array([x.value for x in raty[0]]), np.array([x.error for x in raty[0]])],
                   ax, vmin=2.9, vmax=6.7,
                   mask = ~(megatab[truemitters]['SNRB'] > 3.0) & detections, edgecolor='firebrick', label='',
                  alpha=0.8, alpha_e=0.8)
#     UBs for Y with detections in X
    y_uppers = ratx[2] & ratx[3] & raty[3] & ~raty[2]
    make_scatter([np.array([x.value for x in ratx[0]]), np.array([x.error for x in ratx[0]])], 
                   [np.array([x.value for x in raty[1]])],
                   ax, vmin=2.9, vmax=6.7,
                   mask = (megatab[truemitters]['SNRB'] > 3.0) & y_uppers, edgecolor='slateblue', label='',
                  marker = 6,
                  alpha=0.8, alpha_e=0.8)
    make_scatter([np.array([x.value for x in ratx[0]]), np.array([x.error for x in ratx[0]])], 
                   [np.array([x.value for x in raty[1]])],
                   ax, vmin=2.9, vmax=6.7,
                   mask = ~(megatab[truemitters]['SNRB'] > 3.0) & y_uppers, edgecolor='firebrick', label='',
                  marker = 6,
                  alpha=0.8, alpha_e=0.8)
#     UBs for X with detections in Y
    x_uppers = ~ratx[2] & ratx[3] & raty[3] & raty[2]
    make_scatter([np.array([x.value for x in ratx[1]])], 
                   [np.array([x.value for x in raty[0]]), np.array([x.error for x in raty[0]])],
                   ax, vmin=2.9, vmax=6.7,
                   mask = (megatab[truemitters]['SNRB'] > 3.0) & x_uppers, edgecolor='slateblue', label='',
                  marker=5,
                  alpha=0.8, alpha_e=0.8)
    make_scatter([np.array([x.value for x in ratx[1]])], 
                   [np.array([x.value for x in raty[0]]), np.array([x.error for x in raty[0]])],
                  ax, vmin=2.9, vmax=6.7,
                   mask = ~(megatab[truemitters]['SNRB'] > 3.0) & x_uppers, edgecolor='firebrick', label='',
                  marker=5,
                  alpha=0.8, alpha_e=0.8)
    
    #put in the AGN NLR model grids:
    _la = [linealiases[x] for x in _l]
    
    # Precompute all points for this subplot
    all_x = []
    all_y = []
    
    for _f in nlr_files:
        with fits.open(_f) as nlr_hdul:  # Using context manager for proper file handling
            for _e in nlr_hdul[3:]:
                x_vals = _e.data[_la[0]] / _e.data[_la[1]]
                y_vals = _e.data[_la[2]] / _e.data[_la[3]]
                all_x.extend(x_vals)
                all_y.extend(y_vals)
    
    # Plot all points at once (more efficient than individual scatter calls)
    ax.scatter(all_x, all_y, color='goldenrod', s=16, alpha=0.5, edgecolor='none', label = 'AGN NLR')
    
    #put in the gutkin model grids:
    allx_neb = []
    ally_neb = []
    allz = []
    for fp in glob.glob(f"../emission_models/nebular_emission_gutkin16/nebular_emission_Z0*.txt"):
        if float('0.0'+fp.split('_')[-1].split('.')[0][2:]) > 0.0152:
            continue
        allx_neb.extend(get_gutkin_flux(fp, _l[0][:4]) / get_gutkin_flux(fp, _l[1][:4]))
        ally_neb.extend(get_gutkin_flux(fp, _l[2][:4]) / get_gutkin_flux(fp, _l[3][:4]))
        allz.extend(float('0.'+fp.split('_')[-1].split('.')[0][2:]) * np.ones(np.shape(get_gutkin_flux(fp, _l[2][:4]) / get_gutkin_flux(fp, _l[3][:4]))[0]))
    allx_neb = np.array(allx_neb)
    ally_neb = np.array(ally_neb)
    allz = np.array(allz)
    okx = np.logical_and(lims[i]['xlims'][0] < allx_neb, allx_neb < lims[i]['xlims'][1])
    oky = np.logical_and(lims[i]['ylims'][0] < ally_neb, ally_neb < lims[i]['ylims'][1])
    ax.scatter(allx_neb[okx*oky], ally_neb[okx*oky], color='teal', alpha=0.15, edgecolor='none', label='HII', s=16, zorder=-1)
    ax.set_ylim(*lims[i]['ylims'])
    ax.set_xlim(*lims[i]['xlims'])
    ax.set_xlabel(xlabs[i])
    ax.set_ylabel(ylabs[i])
# Create custom legend handles/labels
legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', label='AGN NLR',
              markerfacecolor='goldenrod', markersize=5, alpha=0.5),
    plt.Line2D([0], [0], marker='o', color='w', label='HII',
              markerfacecolor='teal', markersize=5, alpha=0.5),
    plt.Line2D([0], [0], marker='o', color='w', label=r'DP',
              markerfacecolor='slateblue', markersize=6),
    plt.Line2D([0], [0], marker='o', color='w', label='SP',
              markerfacecolor='firebrick', markersize=6)
]
# Add legend to the last subplot (or choose specific ax)
axs[1,0].legend(
    handles=legend_elements, 
    loc='upper right',
    facecolor='white',  # Inner color
    edgecolor='black',  # Border color
    framealpha=0.7,     # 0=transparent, 1=opaque
    fancybox=True,      # Rounded corners
    frameon=True
)
# fig.savefig(f"/home/james/Documents/my_papers/lensed_laes/paper2/figures/line_ratio_scatter.pdf", bbox_inches='tight')
fig.savefig(f"./plots/line_ratio_scatter.pdf", bbox_inches='tight')

plt.show()
plt.close()

In [ ]:
# Make histograms of the following parameters

params_of_interest = [
    # 'LUM',
    # 'EW',
    # 'FWHM',
    'CVEL'
]

# Which lines do we care about?
lines_of_interest = [
    'SiII1260',
    'CII1334',
    'SiIV1394',
    'CIV1548',
    'HeII1640',
    'OIII1660',
    'SiIII1883',
    'CIII1907',
]

params_to_plot = []
for line in lines_of_interest:
    for param in params_of_interest:
        param_name = f"{param}_{line}"
        params_to_plot.append(param_name)

# Add all ZELDA parameters that aren't errors (note that ERR appears in unusual places in the strings
# so we have to be careful about excluding those)
for col in megatable.colnames:
    if 'ZELDA' in col and 'ERR' not in col and '_ML' not in col:
        params_to_plot.append(col)

# Add a few extras for lyman alpha
extra_lya_params = [
    # 'LYA_EW',
    # 'LYA_LUM',
    # 'LYA_CONT_LUM',
    # 'DISPR',
    # 'BRRATIO',
    # 'BRSEP',
    # 'FWHMR',
    # 'ASYMR',
    # 'DISPB',
    # 'FWHMB',
    # 'ASYMB',
    # 'DELTAV_LYA',
]
for param in extra_lya_params:
    params_to_plot.append(param)

# Finally, absorption information
absorption_params = [
    # 'DV_LI_ABS',
    # 'EW_LI_ABS',
    # 'W_LI_ABS',
    # 'DV_HI_ABS',
    # 'EW_HI_ABS',
    # 'W_HI_ABS',
]
for param in absorption_params:
    params_to_plot.append(param)

# Print summary of which parameters we're plotting
print(f"Plotting histograms for {len(params_to_plot)} parameters:")
for param in params_to_plot:
    print(f" - {param}")

In [ ]:
# Two masks: one for all sources, one for sources with blue peaks
import numpy as np

# General SNR threshold mask
snrr_threshold = 5.0
mask_snrr = (megatable['SNRR'] > snrr_threshold)
mask_snr = (megatable['SNRR'] > snrr_threshold) | (megatable['SNRB'] > snrr_threshold)

# Masks based on the presence of a blue peak (SNRB > 3.0)
mask_all = np.ones(len(megatable), dtype=bool)
mask_blue_peak = (megatable['SNRB'] > 3.0)
mask_no_blue_peak = ~mask_blue_peak
# Masks based on a B/R ratio threshold
threshold = 0.2
br_ratio = megatable['FLUXB'] / megatable['FLUXR']
br_ratio_uplims = 3.0 / (megatable['SNRR'])
high_br_mask = (br_ratio > threshold) & mask_blue_peak
low_br_mask = ((br_ratio <= threshold) & mask_blue_peak) | ((~mask_blue_peak) & (br_ratio_uplims > threshold))
# Masks based on outflow velocity (VEXP_ZELDA)
vexp_threshold = 100
vexp_high_mask = (megatable['VEXP_ZELDA'] > vexp_threshold)
vexp_low_mask = (megatable['VEXP_ZELDA'] <= vexp_threshold)
# Masks based on DELTAV_LYA
deltav_threshold = 150
deltav_high_mask = (megatable['DELTAV_LYA'] > deltav_threshold)
deltav_low_mask = (megatable['DELTAV_LYA'] <= deltav_threshold)


# Which masks to use
mask_a = deltav_low_mask & mask_snrr
mask_b = deltav_high_mask & mask_snrr
# Which labels to use for the masks
label_a = 'Low DELTAV_LYA'
label_b = 'High DELTAV_LYA'

# Print summary of mask statistics
print(f"Total sources: {len(megatable)}")
print(f"Sources with blue peaks (SNRB > 3.0): {np.sum(mask_blue_peak)}")
print(f"Sources without blue peaks (SNRB <= 3.0): {np.sum(mask_no_blue_peak)}")
print(f"Sources with high B/R ratio: {np.sum(high_br_mask)}")
print(f"Sources with low B/R ratio: {np.sum(low_br_mask)}")
print(f"Sources with low DELTAV_LYA (<= {deltav_threshold} km/s): {np.sum(mask_a)}")
print(f"Sources with high DELTAV_LYA (> {deltav_threshold} km/s): {np.sum(mask_b)}")

for param in params_to_plot:
    try:
        print(f"Plotting histogram for {param}...")
        fig, ax = plt.subplots(figsize=(5, 5))
        generate_histogram(param, megatable, ax=ax,
                           colors=['purple'],
                           ks_test=False,
                          abs_lines=['SiII1260', 'CII1334', 'SiIV1394'])
        plt.tight_layout()
        plt.savefig(f"./plots/{param}_histogram.png", dpi=300)
        plt.show()
    except Exception as e:
        print(f"Error plotting histogram for {param}: {e}")